In [1]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  NanoMind-308M — Inference Cell                                          ║
# ║  Auto-downloads latest checkpoint from HuggingFace & generates text      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ═══════════════════════════════════════
#  ▶  EDIT THESE
# ═══════════════════════════════════════
HF_TOKEN    = "api-key"
HF_REPO_ID  = "shawneil/NanoMind-308m"

PROMPT      = "Once upon a time in a land far away"
MAX_TOKENS  = 2048          # up to 2048
TEMPERATURE = 0.8
TOP_K       = 50
TOP_P       = 0.95
REP_PENALTY = 1.1
STREAM      = True          # print tokens as they generate

# ═══════════════════════════════════════
#  Sample prompts to try  (copy any into PROMPT above)
# ═══════════════════════════════════════
SAMPLE_PROMPTS = [
    "Once upon a time in a land far away",
    "The history of artificial intelligence began",
    "In the year 2087, humans discovered",
    "The quantum computer hummed softly as",
    "She opened the ancient manuscript and read",
    "The ocean stretched endlessly before him, and",
    "Scientists have long debated whether consciousness",
    "Deep in the forest, the old wizard",
    "The stock market crashed in an instant when",
    "To understand the universe, one must first",
    "The last human city on Mars was",
    "He had three hours to defuse the situation before",
]

# ══════════════════════════════════════════════════════════════════════
# 0.  Install deps
# ══════════════════════════════════════════════════════════════════════
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "tiktoken", "huggingface_hub",
], check=True)

# ══════════════════════════════════════════════════════════════════════
# 1.  Imports
# ══════════════════════════════════════════════════════════════════════
import os, math, shutil, torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from pathlib import Path
from huggingface_hub import HfApi, hf_hub_download
import tiktoken

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU   : {torch.cuda.get_device_name(0)}")

# ══════════════════════════════════════════════════════════════════════
# 2.  Model definition (self-contained)
# ══════════════════════════════════════════════════════════════════════
@dataclass
class ModelConfig:
    vocab_size:    int   = 50257
    d_model:       int   = 1152
    n_heads:       int   = 16
    n_kv_heads:    int   = 4
    n_layers:      int   = 18
    max_seq_len:   int   = 512
    ff_mult:       int   = 4
    dropout:       float = 0.0
    use_moe:       bool  = False
    num_experts:   int   = 4
    top_k_experts: int   = 2
    rope_theta:    float = 10000.0


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps    = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        x32 = x.float()
        rms = x32.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return (x32 * rms).to(x.dtype).clone() * self.weight


def precompute_freqs_cis(head_dim, max_len, theta=10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t     = torch.arange(max_len, device=freqs.device)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)


def apply_rope(xq, xk, freqs_cis):
    def rot(x, f):
        xc  = torch.view_as_complex(x.float().reshape(*x.shape[:-1], -1, 2))
        out = torch.view_as_real(xc * f[:x.shape[1]].unsqueeze(0).unsqueeze(2)).flatten(3)
        return out.to(x.dtype)
    return rot(xq, freqs_cis), rot(xk, freqs_cis)


class GQAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.nh  = cfg.n_heads
        self.nkv = cfg.n_kv_heads
        self.hd  = cfg.d_model // cfg.n_heads
        self.q   = nn.Linear(cfg.d_model, cfg.n_heads    * self.hd, bias=False)
        self.k   = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.hd, bias=False)
        self.v   = nn.Linear(cfg.d_model, cfg.n_kv_heads * self.hd, bias=False)
        self.o   = nn.Linear(cfg.n_heads * self.hd, cfg.d_model,    bias=False)
    def forward(self, x, freqs_cis):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh,  self.hd)
        k = self.k(x).view(B, T, self.nkv, self.hd)
        v = self.v(x).view(B, T, self.nkv, self.hd)
        q, k = apply_rope(q, k, freqs_cis)
        reps = self.nh // self.nkv
        k = k.repeat_interleave(reps, dim=2)
        v = v.repeat_interleave(reps, dim=2)
        q, k, v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(out.transpose(1,2).contiguous().view(B, T, -1))


class SwiGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        h = int(cfg.d_model * cfg.ff_mult * 2 / 3)
        h = (h + 63) // 64 * 64
        self.w1 = nn.Linear(cfg.d_model, h, bias=False)
        self.w2 = nn.Linear(h, cfg.d_model, bias=False)
        self.w3 = nn.Linear(cfg.d_model, h, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))


class MoEFF(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.top_k   = cfg.top_k_experts
        self.experts = nn.ModuleList([SwiGLU(cfg) for _ in range(cfg.num_experts)])
        self.gate    = nn.Linear(cfg.d_model, cfg.num_experts, bias=False)
    def forward(self, x):
        B, T, D = x.shape
        flat = x.view(-1, D)
        w, idx = torch.topk(F.softmax(self.gate(flat), dim=-1), self.top_k, dim=-1)
        out = torch.zeros_like(flat)
        for i, expert in enumerate(self.experts):
            sel = (idx == i).any(dim=-1)
            if not sel.any(): continue
            rows = sel.nonzero(as_tuple=True)[0]
            kpos = (idx[rows] == i).nonzero(as_tuple=True)[1]
            out[rows] += w[rows, kpos].unsqueeze(-1) * expert(flat[rows])
        return out.view(B, T, D)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.an   = RMSNorm(cfg.d_model)
        self.fn   = RMSNorm(cfg.d_model)
        self.attn = GQAttention(cfg)
        self.ff   = MoEFF(cfg) if cfg.use_moe else SwiGLU(cfg)
    def forward(self, x, fc):
        x = x + self.attn(self.an(x), fc)
        x = x + self.ff(self.fn(x))
        return x


class NanoMind(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg     = cfg
        self.embed   = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.blocks  = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.norm    = RMSNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.embed.weight = self.lm_head.weight
        self.register_buffer(
            "freqs_cis",
            precompute_freqs_cis(cfg.d_model // cfg.n_heads,
                                 cfg.max_seq_len * 2, theta=cfg.rope_theta)
        )

    def rebuild_rope(self, new_seq_len, new_theta):
        self.cfg.max_seq_len = new_seq_len
        self.cfg.rope_theta  = new_theta
        self.register_buffer("freqs_cis",
            precompute_freqs_cis(self.cfg.d_model // self.cfg.n_heads,
                                 new_seq_len * 2, theta=new_theta)
            .to(next(self.parameters()).device))

    def forward(self, idx):
        x  = self.embed(idx)
        fc = self.freqs_cis[:idx.shape[1]]
        for blk in self.blocks:
            x = blk(x, fc)
        return self.lm_head(self.norm(x))

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=2048, temperature=0.8,
                 top_k=50, top_p=0.95, repetition_penalty=1.1, stream=False):
        generated = []
        for _ in range(max_new_tokens):
            ic     = idx[:, -self.cfg.max_seq_len:]
            logits = self.forward(ic)[:, -1, :] / temperature

            if repetition_penalty != 1.0 and generated:
                seen = torch.tensor(generated, device=idx.device, dtype=torch.long)
                logits[0, seen] /= repetition_penalty

            if top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")

            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True)
                cumprobs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                remove = cumprobs - F.softmax(sorted_logits, dim=-1) > top_p
                sorted_logits[remove] = float("-inf")
                logits = torch.zeros_like(logits).scatter_(1, sorted_idx, sorted_logits)

            token = torch.multinomial(F.softmax(logits, dim=-1), 1)
            generated.append(token.item())
            idx = torch.cat([idx, token], dim=1)

            if stream:
                yield token.item()
            if token.item() == 50256:  # EOS
                break

        if not stream:
            return idx

# ══════════════════════════════════════════════════════════════════════
# 3.  HF checkpoint auto-discovery & download
# ══════════════════════════════════════════════════════════════════════
STAGE_PREFIXES = [
    ("checkpoint_ctx4096_step_", 4096, 2),
    ("checkpoint_ctx2048_step_", 2048, 1),
    ("checkpoint_ctx1024_step_", 1024, 0),
    ("checkpoint_step_",          512, -1),
]

def _ntk_chain(stage):
    HD = 1152 // 16
    def _ntk(ol, nl, ot): return ot * ((nl / ol) ** (HD / (HD - 2)))
    CHAIN = [(512, 1024), (1024, 2048), (2048, 4096)]
    theta = 10000.0
    for i in range(stage + 1):
        theta = _ntk(CHAIN[i][0], CHAIN[i][1], theta)
    return theta

def find_latest_hf_checkpoint(repo_id, token):
    api   = HfApi()
    files = [f.path for f in api.list_repo_tree(repo_id=repo_id,
             token=token or None, recursive=True)
             if hasattr(f, "path") and f.path.endswith(".pt")]
    for prefix, seq_len, stage in STAGE_PREFIXES:
        matches = sorted([f for f in files if f.startswith(prefix)],
                         key=lambda x: int(x.replace(".pt","").split("_")[-1]))
        if matches:
            return matches[-1], seq_len, stage
    raise FileNotFoundError(f"No .pt checkpoint found in '{repo_id}'.")

CACHE = Path("/kaggle/working/nanomind_cache")
CACHE.mkdir(parents=True, exist_ok=True)

print("🔍 Scanning HuggingFace for latest checkpoint...")
filename, seq_len, stage = find_latest_hf_checkpoint(HF_REPO_ID, HF_TOKEN)
print(f"✅ Found : {filename}")
print(f"   ctx   : {seq_len} tokens  |  stage: {stage}")

local_path = CACHE / Path(filename).name
if local_path.exists():
    print(f"📦 Already cached: {local_path}")
else:
    print(f"⬇️  Downloading...")
    src = hf_hub_download(repo_id=HF_REPO_ID, filename=filename,
                          token=HF_TOKEN or None, local_dir=str(CACHE))
    if str(src) != str(local_path):
        shutil.copy(src, local_path)
    print(f"✅ Saved to {local_path}")

# ══════════════════════════════════════════════════════════════════════
# 4.  Load model
# ══════════════════════════════════════════════════════════════════════
print("\n🔧 Loading model...")
ckpt      = torch.load(str(local_path), map_location="cpu", weights_only=False)
saved_cfg = ckpt.get("config", {})
theta     = _ntk_chain(stage) if stage >= 0 else 10000.0

cfg = ModelConfig(
    vocab_size  = saved_cfg.get("vocab_size",  50257),
    d_model     = saved_cfg.get("d_model",     1152),
    n_heads     = saved_cfg.get("n_heads",     16),
    n_kv_heads  = saved_cfg.get("n_kv_heads",  4),
    n_layers    = saved_cfg.get("n_layers",    18),
    max_seq_len = seq_len,
    dropout     = 0.0,
    rope_theta  = theta,
)

model  = NanoMind(cfg)
state  = {k.replace("module.", ""): v for k, v in ckpt["model"].items()}
missing, _ = model.load_state_dict(state, strict=False)
real_missing = [k for k in missing if "freqs_cis" not in k]
if real_missing:
    print(f"⚠️  Missing keys: {real_missing}")

model.rebuild_rope(seq_len, theta)
model.eval().to(DEVICE)

params   = sum(p.numel() for p in model.parameters()) / 1e6
ft_step  = ckpt.get("ft_step", ckpt.get("step", "?"))
print(f"✅ Model ready: {params:.1f}M params | seq={seq_len} | θ={theta:.0f} | step={ft_step}")

# ══════════════════════════════════════════════════════════════════════
# 5.  Tokenizer
# ══════════════════════════════════════════════════════════════════════
enc = tiktoken.get_encoding("gpt2")

# ══════════════════════════════════════════════════════════════════════
# 6.  Generate!
# ══════════════════════════════════════════════════════════════════════
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║  Prompt      : {PROMPT[:55]:<55} ║
║  Max tokens  : {MAX_TOKENS:<54} ║
║  Temperature : {TEMPERATURE:<54} ║
║  Top-k / p   : {TOP_K} / {TOP_P:<50} ║
╚══════════════════════════════════════════════════════════════════╝
""")

tokens = enc.encode_ordinary(PROMPT)
idx    = torch.tensor([tokens], dtype=torch.long, device=DEVICE)

if STREAM:
    print(PROMPT, end="", flush=True)
    full = tokens[:]
    for tok in model.generate(idx, max_new_tokens=MAX_TOKENS,
                              temperature=TEMPERATURE, top_k=TOP_K,
                              top_p=TOP_P, repetition_penalty=REP_PENALTY,
                              stream=True):
        full.append(tok)
        print(enc.decode([tok]), end="", flush=True)
    print()
    generated_text = enc.decode(full)
else:
    out            = model.generate(idx, max_new_tokens=MAX_TOKENS,
                                    temperature=TEMPERATURE, top_k=TOP_K,
                                    top_p=TOP_P, repetition_penalty=REP_PENALTY,
                                    stream=False)
    generated_text = enc.decode(out[0].tolist())
    print(generated_text)

n_generated = len(enc.encode_ordinary(generated_text)) - len(tokens)
print(f"\n[Generated {n_generated} tokens]")

Device: cpu
🔍 Scanning HuggingFace for latest checkpoint...
✅ Found : checkpoint_ctx2048_step_127500.pt
   ctx   : 2048 tokens  |  stage: 1
⬇️  Downloading...


checkpoint_ctx2048_step_127500.pt:   0%|          | 0.00/1.24G [00:00<?, ?B/s]

✅ Saved to /kaggle/working/nanomind_cache/checkpoint_ctx2048_step_127500.pt

🔧 Loading model...
✅ Model ready: 308.8M params | seq=2048 | θ=41616 | step=3000

╔══════════════════════════════════════════════════════════════════╗
║  Prompt      : Once upon a time in a land far away                     ║
║  Max tokens  : 2048                                                   ║
║  Temperature : 0.8                                                    ║
║  Top-k / p   : 50 / 0.95                                               ║
╚══════════════════════════════════════════════════════════════════╝

Once upon a time in a land far away from the rest of the universe, there was no way to predict how many times a future disaster would affect the rest of the universe. The idea of an apocalypse or catastrophic event was one such prediction. That is until researchers in Germany came up with a clever device that used this prediction to predict an impending catastrophe.

The idea behind the invention of t